# Air Quality Sensor Calibration - Data Analysis and Machine Learning

This notebook documents a reproducible exploratory analysis and regression workflow for the UCI Air Quality dataset.

The goal is to inspect noisy field sensor data, convert the dataset-specific `-200` missing-value sentinel, build leakage-aware features, and benchmark regression models for reference analyzer targets.

## 1. Setup

Run this notebook from the repository root so the local `src` package can be imported.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_TARGET, REFERENCE_COLUMNS, SENSOR_COLUMNS, SUPPORTED_TARGETS
from src.models.regression import evaluate_models
from src.utils.analysis import (
    build_dataset_profile,
    monthly_missingness,
    numeric_summary,
    sensor_reference_correlation,
    variable_type_summary,
)
from src.utils.data_loader import load_air_quality_data, summarize_data_quality
from src.utils.metadata import variable_dictionary
from src.utils.preprocessing import build_modeling_frame, missingness_table


## 2. Load and Validate the Dataset

The loader handles the original semicolon-separated format, European decimal commas, timestamp parsing, and conversion of `-200` into missing values.

In [ ]:
df = load_air_quality_data()
profile = build_dataset_profile(df)
quality = summarize_data_quality()

profile, quality

In [ ]:
df.head()

## 3. Dataset and Variable Dictionary

The dataset combines certified reference analyzer measurements and raw metal oxide sensor responses. This distinction is important for interpreting the project correctly.

- `GT` columns are ground-truth reference gas concentrations.
- `PT08...` columns are raw sensor responses, not certified pollutant concentrations.
- `CO` means carbon monoxide.
- `C6H6` means benzene.
- `NMHC` means non-methane hydrocarbons.
- `NOx` means total nitrogen oxides.
- `NO2` means nitrogen dioxide.
- `O3` means ozone.


In [ ]:
variable_dictionary()


## 4. Variable Types

After preprocessing, the dataset should mostly contain numerical variables plus the parsed datetime timestamp.

In [ ]:
type_summary = variable_type_summary(df)
type_summary

In [ ]:
px.pie(
    type_summary,
    names="variable_type",
    values="count",
    hole=0.45,
    title="Variable type distribution after preprocessing",
)

## 5. Missing Values

Missing values are scientifically important here because the original dataset uses `-200` as an invalid measurement marker.

In [ ]:
missing = missingness_table(df)
missing

In [ ]:
px.bar(
    missing[missing["missing_values"] > 0],
    x="column",
    y="missing_percent",
    title="Missing values after converting -200 sentinels",
)

In [ ]:
monthly = monthly_missingness(df)
px.line(monthly, x="month", y="missing_cells", markers=True, title="Monthly missing cells")

## 6. Descriptive Statistics and Sensor Relationships

In [ ]:
numeric_summary(df)

In [ ]:
corr = sensor_reference_correlation(df, method="pearson")
px.imshow(corr, text_auto=".2f", aspect="auto", title="Pearson correlation matrix: sensors and reference gases")

In [ ]:
sensor = "PT08.S2(NMHC)"
target = DEFAULT_TARGET
scatter_df = df[[sensor, target]].dropna()
px.scatter(
    scatter_df,
    x=sensor,
    y=target,
    opacity=0.35,
    title=f"{sensor} vs {target}",
)

## 7. Leakage-Aware Modelling Frame

The modelling frame uses sensor responses, meteorological variables, and temporal features. It intentionally excludes other reference analyzer gas concentrations to keep the task closer to sensor calibration.

In [ ]:
X, y = build_modeling_frame(df, DEFAULT_TARGET)
X.head(), y.head(), X.shape, y.shape

In [ ]:
X.columns.tolist()

## 8. Regression Benchmark

The benchmark uses a chronological holdout split and time-series cross-validation on the training period. This is more defensible than a purely random split for a dataset affected by time and drift.

In [ ]:
run = evaluate_models(df, target=DEFAULT_TARGET, test_size=0.2, cv_splits=5)
run.leaderboard

In [ ]:
run.best_model_name, run.train_rows, run.test_rows

## 9. Interpretation

This workflow demonstrates a structured baseline for scientific sensor data analysis. It handles dataset-specific missing values, keeps the regression task explicit, avoids obvious target leakage, and evaluates models with temporal structure in mind.

It does not solve drift correction, uncertainty quantification, calibration transfer, real-time monitoring, or publication-level model validation.